# 01 · Exploratory Data Analysis (EDA)

**Proyek:** Prediksi Risiko Diabetes Menggunakan Multilayer Perceptron (MLP)
**Penanggung jawab:** Afriyoga Haganta Tarigan (bagian ketidakseimbangan kelas: Johannes Hartanto)
**Prasyarat:** `00_setup.ipynb` sudah berhasil dijalankan.

## Deskripsi
Notebook ini menelusuri karakteristik dataset BRFSS2015 sebelum preprocessing dan pemodelan. Setiap temuan diarahkan untuk menjawab satu pertanyaan praktis: **apa dampaknya terhadap langkah preprocessing dan desain model?**

## Pemetaan ke format laporan (Bab A.3)

| Butir format | Bagian notebook |
|---|---|
| 3.a Distribusi label/kelas | Bagian 5 |
| 3.b Word count | Bagian 6 (tidak berlaku, dijelaskan alasannya) |
| 3.c Histogram fitur numerik | Bagian 7 dan 8 |
| 3.d Korelasi antarfitur | Bagian 9 |
| 3.e Contoh sampel per kelas | Bagian 10 |
| 3.f Temuan penting untuk preprocessing | Bagian 11 |

Bagian 3 dan 4 (kamus fitur, kualitas data, duplikat) tidak diminta secara eksplisit oleh butir 3, tetapi menjadi bahan untuk bab **A.1 Kompleksitas masalah**, **A.2.d Kompleksitas dataset**, dan **A.4.a Pembersihan data**.

> **Prinsip:** notebook ini hanya membaca data. Tidak ada baris yang dihapus atau diubah di sini. Semua keputusan pembersihan diambil dan diterapkan di `02_preprocessing.ipynb`.

## 1. Bootstrap Proyek

In [ ]:
# ============================================================
# Bootstrap: menyiapkan path proyek (Colab maupun lokal)
# ============================================================
import os
import subprocess
import sys
from pathlib import Path

# GANTI sekali saja dengan URL repository GitHub kelompok (harus PUBLIK)
REPO_URL = "https://github.com/GANTI-USERNAME/diabetes-risk-mlp.git"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = Path("/content/diabetes-risk-mlp")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=False)
else:
    # Lokal: naik folder sampai menemukan folder src/
    ROOT = Path.cwd().resolve()
    while not (ROOT / "src").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Berjalan di Colab :", IN_COLAB)
print("Root proyek       :", ROOT)

## 2. Import Library dan Konfigurasi

**Alasan pemilihan alat:**
- `pandas` dan `numpy` untuk manipulasi data tabular.
- `matplotlib` dan `seaborn` untuk visualisasi. Semua grafik disimpan ke `results/figures/` dengan awalan `eda_` agar mudah dirujuk di laporan.
- `scipy.stats` untuk skewness.
- `mutual_info_classif` dari scikit-learn untuk mengukur keterkaitan non-linear antara fitur dan target, sebagai pelengkap korelasi.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.feature_selection import mutual_info_classif

from src.config import (BINARY_FEATURES, FEATURE_DESCRIPTIONS, FEATURES,
                        NUMERIC_FEATURES, ORDINAL_FEATURES, RESULTS_DIR, SEED,
                        TARGET, VALID_RANGES)
from src.data import load_raw
from src.utils import save_fig, set_seed

set_seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

CLASS_NAMES = {0: "Tidak diabetes", 1: "Prediabetes/Diabetes"}
PALETTE = {0: "#4C72B0", 1: "#DD8452"}

# Angka penting dikumpulkan di sini lalu disimpan ke results/eda_summary.json
summary = {}

## 3. Memuat Data dan Kamus Fitur

**Tujuan:** memuat data yang sudah tervalidasi lalu menampilkan ringkasan setiap fitur (kelompok, rentang, jumlah nilai unik).

**Alasan:** pengelompokan fitur menjadi biner, ordinal, dan numerik menentukan cara preprocessing. Fitur ordinal seperti `Age` (1 sampai 13) bukan angka kontinu, melainkan kategori yang punya urutan.

In [ ]:
df = load_raw()
X, y = df[FEATURES], df[TARGET]
print(f"Ukuran data: {df.shape[0]:,} baris x {df.shape[1]} kolom")

def feature_group(feature):
    if feature in BINARY_FEATURES:
        return "Biner"
    if feature in ORDINAL_FEATURES:
        return "Ordinal"
    return "Numerik"

feature_table = pd.DataFrame({
    "Kelompok": [feature_group(f) for f in FEATURES],
    "Rentang codebook": [f"{VALID_RANGES[f][0]}-{VALID_RANGES[f][1]}" for f in FEATURES],
    "Min": X.min().values,
    "Maks": X.max().values,
    "Nilai unik": X.nunique().values,
    "Keterangan": [FEATURE_DESCRIPTIONS[f] for f in FEATURES],
}, index=FEATURES)

summary["n_rows"], summary["n_features"] = int(df.shape[0]), len(FEATURES)
summary["feature_groups"] = feature_table["Kelompok"].value_counts().to_dict()
feature_table

**Cara membaca:** pastikan kolom *Min* dan *Maks* berada di dalam *Rentang codebook*. Perhatikan juga jumlah nilai unik `BMI`, karena fitur ini satu-satunya yang benar-benar mendekati kontinu.

## 4. Kualitas Data: Missing Value, Nilai Tidak Valid, dan Duplikat

**Tujuan:** memeriksa tiga masalah kualitas data yang paling umum sebelum preprocessing.

**Alasan khusus untuk duplikat:** hampir semua fitur di dataset ini biner atau ordinal. Akibatnya, dua responden berbeda bisa saja memberikan jawaban yang persis sama. Jadi duplikat **belum tentu** data kotor. Yang perlu diukur:
1. **Duplikat persis**: fitur dan label sama.
2. **Profil fitur identik dengan label berbeda (konflik label)**: secara informasi, model tidak mungkin membedakan baris-baris ini. Ini memberi batas atas performa yang bisa dicapai model apa pun.

In [ ]:
# 4.1 Missing value dan nilai di luar rentang codebook
quality = pd.DataFrame({
    "Missing": df.isna().sum(),
    "Di luar rentang": pd.Series({c: int((~df[c].between(*VALID_RANGES.get(c, (0, 1)))).sum())
                                  for c in df.columns}),
})
summary["missing_total"] = int(quality["Missing"].sum())
summary["out_of_range_total"] = int(quality["Di luar rentang"].sum())
print("Total missing value      :", summary["missing_total"])
print("Total nilai di luar rentang:", summary["out_of_range_total"])
quality.T

In [ ]:
# 4.2 Duplikat persis dan konflik label
dup_full_mask = df.duplicated(keep="first")            # baris kedua dst. yang persis sama
dup_feature_mask = df.duplicated(subset=FEATURES, keep="first")

profile_stats = df.groupby(FEATURES, sort=False)[TARGET].agg(["nunique", "size", "mean"])
conflict = profile_stats[profile_stats["nunique"] > 1]

summary["duplicates_exact"] = int(dup_full_mask.sum())
summary["duplicates_exact_pct"] = float(dup_full_mask.mean())
summary["unique_feature_profiles"] = int(len(profile_stats))
summary["conflict_profiles"] = int(len(conflict))
summary["conflict_rows"] = int(conflict["size"].sum())
summary["conflict_rows_pct"] = float(conflict["size"].sum() / len(df))

dup_report = pd.DataFrame({
    "Jumlah": [len(df), summary["unique_feature_profiles"], summary["duplicates_exact"],
               int(dup_feature_mask.sum()), summary["conflict_profiles"], summary["conflict_rows"]],
}, index=["Total baris",
          "Profil fitur unik",
          "Duplikat persis (fitur + label sama)",
          "Duplikat berdasarkan fitur saja",
          "Profil fitur dengan label berbeda (konflik)",
          "Baris yang terlibat konflik label"])
dup_report["% dari total baris"] = dup_report["Jumlah"] / len(df) * 100
dup_report

In [ ]:
# 4.3 Apakah duplikat menumpuk di salah satu kelas
dup_by_class = pd.DataFrame({
    "Jumlah baris": y.value_counts().sort_index(),
    "Duplikat persis": dup_full_mask.groupby(y).sum(),
})
dup_by_class["% duplikat di kelas ini"] = dup_by_class["Duplikat persis"] / dup_by_class["Jumlah baris"] * 100
dup_by_class.index = [CLASS_NAMES[i] for i in dup_by_class.index]
summary["duplicates_pct_by_class"] = {k: float(v) for k, v in dup_by_class["% duplikat di kelas ini"].items()}
dup_by_class

In [ ]:
# 4.4 Lima profil jawaban yang paling sering muncul
top_profiles = (profile_stats.sort_values("size", ascending=False).head(5)
                .reset_index()
                .rename(columns={"size": "Frekuensi", "mean": "Proporsi label 1", "nunique": "Label unik"}))
top_profiles.T

**Cara membaca dan dampaknya:**

- **Missing value dan nilai tidak valid.** Jika keduanya nol, tidak perlu imputasi ataupun koreksi nilai. Tulis hal ini secara eksplisit di laporan (A.4.d), karena "tidak ada missing value" juga merupakan temuan.
- **Duplikat persis.** Kalau proporsinya besar, ada risiko baris identik jatuh di data train dan test sekaligus. Model lalu terlihat bagus karena "menghafal", bukan karena belajar pola. Ada dua opsi yang akan diputuskan di `02_preprocessing.ipynb` (dicatat sebagai keputusan D-07):
  1. Menghapus duplikat persis sebelum split.
  2. Mempertahankan semua baris, tetapi memakai split berbasis grup sehingga satu profil hanya muncul di satu subset.
- **Duplikat per kelas.** Jika duplikat jauh lebih banyak di kelas 0, menghapusnya akan sedikit menggeser proporsi kelas. Pergeseran ini perlu dilaporkan.
- **Konflik label.** Baris dengan jawaban survei identik tetapi label berbeda adalah *noise* yang tidak bisa dihilangkan oleh model apa pun. Angka ini argumen kuat untuk bab **A.1.a Mengapa masalah ini kompleks**, sekaligus menjelaskan mengapa performa tidak akan mendekati 100%.

## 5. Distribusi Label (Butir 3.a)

**Tujuan:** mengukur seberapa tidak seimbang kelas target.

**Alasan:** pada data tidak seimbang, *accuracy* menipu. Model yang selalu menebak kelas mayoritas sudah mendapat accuracy sebesar proporsi kelas mayoritas tanpa belajar apa pun. Angka ini dihitung di bawah sebagai **baseline mayoritas** yang wajib dikalahkan secara bermakna.

In [ ]:
counts = y.value_counts().sort_index()
proportions = counts / counts.sum()

summary["class_counts"] = {CLASS_NAMES[k]: int(v) for k, v in counts.items()}
summary["positive_rate"] = float(proportions[1])
summary["imbalance_ratio"] = float(counts[0] / counts[1])
summary["majority_baseline_accuracy"] = float(proportions.max())

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar([CLASS_NAMES[i] for i in counts.index], counts.values,
              color=[PALETTE[i] for i in counts.index])
for bar, n, p in zip(bars, counts.values, proportions.values):
    ax.text(bar.get_x() + bar.get_width() / 2, n, f"{n:,}\n({p:.1%})",
            ha="center", va="bottom", fontsize=11)
ax.set_title("Distribusi Kelas Target Diabetes_binary")
ax.set_ylabel("Jumlah responden")
ax.set_ylim(0, counts.max() * 1.18)
save_fig(fig, "eda_01_distribusi_kelas")
plt.show()

print(f"Proporsi kelas positif       : {summary['positive_rate']:.2%}")
print(f"Rasio kelas 0 : kelas 1      : {summary['imbalance_ratio']:.2f} : 1")
print(f"Accuracy baseline mayoritas  : {summary['majority_baseline_accuracy']:.2%}")

**Cara membaca dan dampaknya:**
- Angka *accuracy baseline mayoritas* adalah batas bawah. Model MLP yang accuracy-nya hanya sedikit di atas angka ini **belum tentu** berguna. Karena itu metrik utama proyek ini adalah **recall kelas positif, F1-score, dan PR-AUC**, bukan accuracy.
- Rasio ketidakseimbangan menjadi dasar fitur unik 1 (class weight atau focal loss serta penentuan threshold).
- Split train/validation/test wajib **stratified** agar proporsi kelas positif sama di ketiga subset.

## 6. Word Count (Butir 3.b)

**Tidak berlaku.** Butir ini khusus untuk data teks. Seluruh fitur pada dataset ini berupa bilangan (biner, ordinal, dan numerik), sehingga tidak ada teks yang perlu dihitung jumlah katanya. Di laporan, tuliskan kalimat serupa supaya penguji tahu butir ini tidak terlewat.

## 7. Distribusi Fitur Numerik (Butir 3.c)

**Tujuan:** melihat bentuk distribusi `BMI`, `MentHlth`, dan `PhysHlth`, serta perbedaannya antarkelas.

**Alasan teknis:**
- Histogram ditampilkan dengan `stat="density"` dan `common_norm=False`. Tanpa normalisasi per kelas, kelas 1 yang jauh lebih sedikit akan tampak "kecil" dan perbedaan bentuknya tidak terlihat.
- `MentHlth` dan `PhysHlth` adalah hitungan hari (0 sampai 30), jadi diplot sebagai data diskret.
- Skewness dihitung karena distribusi yang sangat miring memengaruhi pilihan scaling.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, feature in zip(axes, NUMERIC_FEATURES):
    sns.histplot(data=df, x=feature, hue=TARGET, stat="density", common_norm=False,
                 discrete=(feature != "BMI"), bins=(60 if feature == "BMI" else "auto"),
                 palette=PALETTE, alpha=0.5, element="step", ax=ax)
    ax.set_title(f"Distribusi {feature} per kelas")
    handles = ax.get_legend().legend_handles
    ax.legend(handles, [CLASS_NAMES[0], CLASS_NAMES[1]], title=None)
fig.tight_layout()
save_fig(fig, "eda_02_histogram_numerik")
plt.show()

numeric_by_class = df.groupby(TARGET)[NUMERIC_FEATURES].agg(["mean", "median", "std"]).T
numeric_by_class.columns = [CLASS_NAMES[c] for c in numeric_by_class.columns]
numeric_by_class

In [ ]:
# Skewness dan penumpukan nilai (heaping) pada data hitungan hari
numeric_shape = pd.DataFrame({
    "Skewness": [stats.skew(df[f]) for f in NUMERIC_FEATURES],
    "% bernilai 0": [(df[f] == 0).mean() * 100 for f in NUMERIC_FEATURES],
    "% nilai maksimum": [(df[f] == df[f].max()).mean() * 100 for f in NUMERIC_FEATURES],
}, index=NUMERIC_FEATURES)

# Heaping: dari responden yang menjawab > 0 hari, berapa persen menjawab kelipatan 5
for f in ["MentHlth", "PhysHlth"]:
    positive_days = df.loc[df[f] > 0, f]
    numeric_shape.loc[f, "% kelipatan 5 (dari >0)"] = (positive_days % 5 == 0).mean() * 100

summary["numeric_shape"] = numeric_shape.round(4).to_dict(orient="index")
numeric_shape

**Cara membaca dan dampaknya:**
- **Perbedaan antarkelas.** Jika kurva kelas 1 bergeser ke kanan pada `BMI` dan `PhysHlth`, fitur tersebut kemungkinan informatif. Bandingkan juga dengan tabel mean/median.
- **Skewness tinggi dan banyak nilai 0** pada `MentHlth` dan `PhysHlth` wajar untuk data hitungan hari. Sebagian besar responden tidak mengalami hari buruk sama sekali.
- **Heaping.** Jika persentase kelipatan 5 jauh di atas yang diharapkan secara acak (sekitar 20% untuk rentang 1 sampai 30), responden cenderung membulatkan jawaban. Ini ciri data *self-report* dan layak disebut sebagai sumber noise di bab A.1 dan A.2.d.
- **Dampak ke preprocessing.** Fitur numerik perlu di-*standardize* agar skalanya setara dengan fitur biner. Scaler wajib di-*fit* hanya pada data train.

## 8. Fitur Ordinal, Fitur Biner, dan Outlier (Butir 3.c lanjutan)

**Tujuan:** untuk data tabular berisi banyak fitur kategorikal, histogram biasa kurang informatif. Yang lebih berguna adalah **prevalensi kelas positif** pada tiap nilai fitur, yaitu P(label = 1 | nilai fitur).

**Alasan:** jika prevalensi naik secara konsisten seiring kategori (misalnya semakin tua usia, semakin tinggi prevalensi), maka urutan kategori bermakna dan fitur ordinal **layak diperlakukan sebagai angka berurutan**, tidak perlu one-hot encoding.

In [ ]:
# 8.1 Prevalensi kelas positif per kategori fitur ordinal
overall_rate = y.mean()
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
ordinal_trend = {}
for ax, feature in zip(axes.ravel(), ORDINAL_FEATURES):
    rate = df.groupby(feature)[TARGET].mean()
    n = df[feature].value_counts().sort_index()
    ax.bar(rate.index.astype(str), rate.values, color=PALETTE[1], alpha=0.85)
    ax.axhline(overall_rate, ls="--", color="gray", label=f"Rata-rata ({overall_rate:.1%})")
    ax.set_title(f"Prevalensi label 1 menurut {feature}")
    ax.set_xlabel(feature)
    ax.set_ylabel("P(label = 1)")
    ax.set_ylim(0, max(rate.max(), overall_rate) * 1.3)
    ax.legend(loc="upper left")
    # Korelasi Spearman antara urutan kategori dan prevalensinya (uji monotonic)
    ordinal_trend[feature] = {
        "spearman_kategori_vs_prevalensi": float(stats.spearmanr(rate.index, rate.values).statistic),
        "kategori_paling_jarang": int(n.idxmin()), "n_paling_jarang": int(n.min()),
    }
fig.tight_layout()
save_fig(fig, "eda_03_prevalensi_ordinal")
plt.show()

summary["ordinal_trend"] = ordinal_trend
pd.DataFrame(ordinal_trend).T

In [ ]:
# 8.2 Prevalensi kelas positif untuk fitur biner (nilai 1 vs nilai 0)
binary_rows = []
for feature in BINARY_FEATURES:
    rate1 = y[df[feature] == 1].mean()
    rate0 = y[df[feature] == 0].mean()
    binary_rows.append({"Fitur": feature, "P(y=1 | x=1)": rate1, "P(y=1 | x=0)": rate0,
                        "Selisih": rate1 - rate0, "% responden x=1": df[feature].mean() * 100})
binary_table = pd.DataFrame(binary_rows).set_index("Fitur").sort_values("Selisih")

fig, ax = plt.subplots(figsize=(9, 6.5))
colors = [PALETTE[1] if d > 0 else PALETTE[0] for d in binary_table["Selisih"]]
ax.barh(binary_table.index, binary_table["Selisih"], color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Selisih prevalensi label 1 saat fitur biner bernilai 1 dibanding 0")
ax.set_xlabel("P(y=1 | x=1) - P(y=1 | x=0)")
fig.tight_layout()
save_fig(fig, "eda_04_prevalensi_biner")
plt.show()

summary["binary_prevalence_diff"] = binary_table["Selisih"].round(4).to_dict()
binary_table.sort_values("Selisih", ascending=False)

In [ ]:
# 8.3 Outlier pada fitur numerik (aturan IQR 1,5)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
outlier_rows = {}
for ax, feature in zip(axes, NUMERIC_FEATURES):
    sns.boxplot(data=df, x=TARGET, y=feature, hue=TARGET, palette=PALETTE,
                legend=False, ax=ax, fliersize=1.5)
    ax.set_xticks([0, 1], [CLASS_NAMES[0], CLASS_NAMES[1]])
    ax.set_xlabel("")
    ax.set_title(f"Boxplot {feature}")
    q1, q3 = df[feature].quantile([0.25, 0.75])
    upper = q3 + 1.5 * (q3 - q1)
    outlier_rows[feature] = {"Q1": q1, "Q3": q3, "Batas atas IQR": upper,
                             "Jumlah outlier": int((df[feature] > upper).sum()),
                             "% outlier": float((df[feature] > upper).mean() * 100)}
fig.tight_layout()
save_fig(fig, "eda_05_boxplot_outlier")
plt.show()

outlier_table = pd.DataFrame(outlier_rows).T
summary["outliers_iqr"] = outlier_table.round(4).to_dict(orient="index")
summary["bmi_extreme"] = {f"BMI>={t}": int((df['BMI'] >= t).sum()) for t in (50, 60, 70, 80)}
print("Jumlah BMI ekstrem:", summary["bmi_extreme"])
outlier_table

**Cara membaca dan dampaknya:**
- **Fitur ordinal.** Nilai Spearman antara urutan kategori dan prevalensinya yang mendekati +1 atau -1 menunjukkan hubungan monoton. Ini argumen untuk memperlakukan `GenHlth`, `Age`, `Education`, dan `Income` sebagai angka berurutan lalu di-*standardize*, alih-alih one-hot encoding yang menambah dimensi tanpa banyak manfaat.
- **Fitur biner.** Selisih positif berarti responden dengan kondisi tersebut lebih sering berada di kelas 1. Fitur dengan selisih paling besar menjadi kandidat faktor risiko utama, yang nanti dibandingkan dengan hasil explainability (fitur unik 2).
- **Outlier.** Aturan IQR pada `MentHlth` dan `PhysHlth` akan menandai banyak nilai sebagai outlier hanya karena mayoritas responden menjawab 0. Itu **bukan** kesalahan data. Untuk `BMI`, nilai yang sangat tinggi secara klinis masih mungkin terjadi, tetapi nilai ekstrem (misalnya di atas 70) patut dicurigai sebagai salah input. Opsinya, dipertahankan atau di-*clip* di batas tertentu. Keputusan diambil di `02_preprocessing.ipynb` dan **tidak boleh** berupa penghapusan baris tanpa alasan.

## 9. Korelasi Antarfitur (Butir 3.d)

**Tujuan:** mengukur hubungan antarfitur dan hubungan setiap fitur dengan target.

**Alasan pemilihan metode:**
- **Spearman** dipakai, bukan Pearson, karena sebagian besar fitur berupa ordinal atau biner dan distribusi numeriknya miring. Spearman mengukur hubungan monoton berbasis peringkat sehingga lebih tepat untuk kondisi ini.
- **Mutual information** ditambahkan karena korelasi hanya menangkap hubungan monoton. Mutual information dapat menangkap keterkaitan non-linear yang justru dimanfaatkan oleh MLP.

In [ ]:
spearman = df[[TARGET] + FEATURES].corr(method="spearman")

fig, ax = plt.subplots(figsize=(15, 12))
mask = np.triu(np.ones_like(spearman, dtype=bool), k=1)
sns.heatmap(spearman, mask=mask, cmap="RdBu_r", vmin=-1, vmax=1, center=0,
            annot=True, fmt=".2f", annot_kws={"size": 7}, square=True,
            cbar_kws={"shrink": 0.7, "label": "Spearman rho"}, ax=ax)
ax.set_title("Matriks Korelasi Spearman (termasuk target)")
fig.tight_layout()
save_fig(fig, "eda_06_korelasi_spearman")
plt.show()

In [ ]:
# 9.1 Pasangan fitur dengan korelasi kuat (|rho| >= 0,4)
pairs = (spearman.loc[FEATURES, FEATURES]
         .where(np.triu(np.ones((len(FEATURES), len(FEATURES)), dtype=bool), k=1))
         .stack().rename("rho").reset_index())
pairs.columns = ["Fitur A", "Fitur B", "rho"]
strong_pairs = pairs[pairs["rho"].abs() >= 0.4].sort_values("rho", key=abs, ascending=False)
summary["strong_feature_pairs"] = strong_pairs.round(4).to_dict(orient="records")
strong_pairs.reset_index(drop=True)

In [ ]:
# 9.2 Keterkaitan fitur dengan target: Spearman dan mutual information
target_corr = spearman.loc[FEATURES, TARGET]
mi = mutual_info_classif(X, y, discrete_features=True, random_state=SEED)
association = pd.DataFrame({"Spearman dengan target": target_corr,
                            "Mutual information": mi}, index=FEATURES)
association["Peringkat Spearman (|rho|)"] = association["Spearman dengan target"].abs().rank(ascending=False).astype(int)
association["Peringkat MI"] = association["Mutual information"].rank(ascending=False).astype(int)
association = association.sort_values("Mutual information", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
order = association.index
axes[0].barh(order, association["Spearman dengan target"], color=PALETTE[1])
axes[0].set_title("Korelasi Spearman fitur dengan target")
axes[0].invert_yaxis()
axes[1].barh(order, association["Mutual information"], color=PALETTE[0])
axes[1].set_title("Mutual information fitur dengan target")
axes[1].invert_yaxis()
fig.tight_layout()
save_fig(fig, "eda_07_asosiasi_target")
plt.show()

summary["target_association_top5_mi"] = association.head(5).index.tolist()
summary["target_association"] = association[["Spearman dengan target", "Mutual information"]].round(5).to_dict(orient="index")
association

**Cara membaca dan dampaknya:**
- **Pasangan berkorelasi kuat.** Jika muncul pasangan seperti kesehatan umum dengan kesehatan fisik, atau pendidikan dengan pendapatan, itu menandakan informasi yang tumpang tindih. MLP relatif toleran terhadap multikolinearitas, sehingga **tidak ada fitur yang perlu dibuang** hanya karena alasan ini. Namun, saat membaca hasil explainability (fitur unik 2), kontribusi dua fitur yang berkorelasi bisa "berbagi" sehingga perlu ditafsirkan dengan hati-hati.
- **Keterkaitan dengan target.** Bandingkan peringkat Spearman dan MI. Fitur yang peringkatnya jauh lebih tinggi di MI mengindikasikan hubungan non-linear. Ini salah satu argumen pemilihan MLP dibanding model linear.
- **Korelasi lemah secara umum.** Jika nilai korelasi tertinggi dengan target hanya berada di kisaran lemah sampai sedang, berarti tidak ada satu fitur pun yang cukup untuk memprediksi sendirian. Model harus menggabungkan banyak sinyal lemah. Ini poin bagus untuk bab A.1.

## 10. Contoh Sampel Data per Kelas (Butir 3.e)

**Tujuan:** format laporan meminta contoh gambar per kelas untuk data citra. Padanannya untuk data tabular adalah contoh baris dari masing-masing kelas.

**Catatan:** sampel diambil acak dengan `random_state=SEED` sehingga baris yang sama muncul di setiap eksekusi dan bisa dikutip di laporan.

In [ ]:
samples = (df.groupby(TARGET, group_keys=False)
             .sample(n=5, random_state=SEED)
             .rename(index=lambda i: f"baris {i}"))
samples.insert(0, "Kelas", samples[TARGET].map(CLASS_NAMES))
samples.drop(columns=TARGET).T

## 11. Ringkasan Temuan dan Implikasi untuk Preprocessing (Butir 3.f)

**Tujuan:** merangkum angka-angka di atas menjadi satu tabel temuan beserta implikasinya. Tabel ini dibangun **otomatis dari hasil eksekusi**, jadi isinya selalu sesuai dengan data aktual.

In [ ]:
pairs_text = ", ".join(f"{r['Fitur A']}-{r['Fitur B']} ({r['rho']:.2f})"
                       for r in summary["strong_feature_pairs"][:4]) or "tidak ada"

findings = pd.DataFrame([
    ["Ketidakseimbangan kelas",
     f"{summary['positive_rate']:.1%} kelas positif, rasio {summary['imbalance_ratio']:.1f}:1",
     "Split stratified. Class weight/focal loss dan tuning threshold (fitur unik 1). Metrik utama recall, F1, PR-AUC."],
    ["Baseline mayoritas",
     f"Accuracy {summary['majority_baseline_accuracy']:.1%} tanpa belajar apa pun",
     "Accuracy tidak dijadikan metrik utama. Semua model dibandingkan dengan baseline ini."],
    ["Missing value",
     f"{summary['missing_total']} sel",
     "Tidak perlu imputasi." if summary["missing_total"] == 0 else "Perlu strategi imputasi."],
    ["Nilai di luar rentang codebook",
     f"{summary['out_of_range_total']} sel",
     "Tidak ada koreksi nilai." if summary["out_of_range_total"] == 0 else "Perlu ditelusuri dan dikoreksi."],
    ["Duplikat persis",
     f"{summary['duplicates_exact']:,} baris ({summary['duplicates_exact_pct']:.1%})",
     "Keputusan D-07: hapus sebelum split, atau split berbasis grup. Cek tidak ada profil identik lintas subset."],
    ["Konflik label",
     f"{summary['conflict_rows']:,} baris ({summary['conflict_rows_pct']:.1%}) pada {summary['conflict_profiles']:,} profil",
     "Noise tak tereduksi. Dibahas sebagai sumber kompleksitas (A.1.a) dan batas atas performa."],
    ["Fitur ordinal",
     "Lihat Spearman kategori vs prevalensi di bagian 8.1",
     "Diperlakukan sebagai numerik berurutan lalu di-standardize, tanpa one-hot (jika tren monoton)."],
    ["Distribusi numerik",
     f"Skewness BMI {summary['numeric_shape']['BMI']['Skewness']:.2f}, "
     f"MentHlth {summary['numeric_shape']['MentHlth']['Skewness']:.2f}, "
     f"PhysHlth {summary['numeric_shape']['PhysHlth']['Skewness']:.2f}",
     "StandardScaler di-fit pada train saja. Nilai 0 yang dominan bukan outlier."],
    ["BMI ekstrem",
     f"{summary['bmi_extreme']['BMI>=70']:,} baris BMI >= 70",
     "Pertahankan atau clip (keputusan di 02). Tidak menghapus baris tanpa alasan."],
    ["Fitur saling berkorelasi",
     pairs_text,
     "Tidak ada fitur dibuang. Diperhatikan saat interpretasi explainability."],
    ["Fitur paling terkait target (MI)",
     ", ".join(summary["target_association_top5_mi"]),
     "Dibandingkan dengan hasil Integrated Gradients (fitur unik 2) sebagai sanity check."],
], columns=["Temuan", "Angka aktual", "Implikasi untuk preprocessing/model"])

with pd.option_context("display.max_colwidth", 120):
    display(findings)

In [ ]:
def to_builtin(obj):
    """Mengubah tipe NumPy menjadi tipe bawaan Python supaya bisa disimpan sebagai JSON."""
    if isinstance(obj, dict):
        return {str(k): to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_builtin(v) for v in obj]
    if isinstance(obj, np.generic):
        return obj.item()
    return obj

summary_path = RESULTS_DIR / "eda_summary.json"
summary_path.write_text(json.dumps(to_builtin(summary), indent=2, ensure_ascii=False))
findings.to_csv(RESULTS_DIR / "eda_findings.csv", index=False)
print("Tersimpan:", summary_path)
print("Tersimpan:", RESULTS_DIR / "eda_findings.csv")
print("Grafik   :", sorted(p.name for p in (RESULTS_DIR / "figures").glob("eda_*.png")))

In [ ]:
# Mengemas results/ menjadi zip supaya grafik bisa diunggah ke branch GitHub masing-masing
import shutil
from src.config import RESULTS_DIR

archive = shutil.make_archive(str(ROOT / "results_export"), "zip", RESULTS_DIR)
print("Arsip dibuat:", archive)

if IN_COLAB:
    from google.colab import files
    files.download(archive)

## 12. Kesimpulan Tahapan

EDA ini menghasilkan tiga kelompok keluaran:
1. **Grafik** di `results/figures/eda_*.png` untuk bab A.3 laporan.
2. **Ringkasan angka** di `results/eda_summary.json` sebagai sumber tunggal angka EDA di laporan Tugas 1 dan UTS.
3. **Tabel temuan** di `results/eda_findings.csv` yang menjadi dasar keputusan di `02_preprocessing.ipynb`.

Interpretasi akhir **sengaja tidak ditulis sebagai fakta di notebook ini** sebelum angkanya terlihat. Setelah notebook dijalankan, interpretasi final ditulis di laporan berdasarkan angka aktual.

**Kirimkan ke mentor setelah menjalankan notebook ini:**
1. Isi `results/eda_summary.json`.
2. Output tabel temuan di bagian 11.
3. Screenshot grafik `eda_01` sampai `eda_07` (atau file zip dari sel unduhan).

Dari situ kita putuskan D-07 (duplikat), perlakuan BMI ekstrem, dan proporsi split, lalu lanjut ke `02_preprocessing.ipynb`.

## Referensi
- Teboul, A. *Diabetes Health Indicators Dataset*. Kaggle. https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset
- UCI Machine Learning Repository. *CDC Diabetes Health Indicators* [Dataset]. https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators
- Xie, Z., Nikolayeva, O., Luo, J., & Li, D. (2019). Building Risk Prediction Models for Type 2 Diabetes Using Machine Learning Techniques. *Preventing Chronic Disease, 16*, E130. https://doi.org/10.5888/pcd16.190109
- Centers for Disease Control and Prevention. *BRFSS 2015 Survey Data and Documentation*. https://www.cdc.gov/brfss/annual_data/annual_2015.html
- scikit-learn. *mutual_info_classif*. https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.mutual_info_classif.html